# Simulation-time analysis (timing trial, disc 244)

(Hof, 2026-11-12) Wall-clock cost of a run, from the per-step **timing CSV** written by `run_model_Hof.py` when `simulation.stream_timing = true` (change log, "sim-time step 1"). The trial runs the same disc (v5 disc 244, the median disc) to **3 Myr and 5 Myr** in parallel (`run_add-planets_pilot_Hof.sh`, Slurm array tasks 0 and 1). Both share every snapshot up to 3 Myr, so they take identical timesteps there and differ only in the ETA they predict.

Goals: estimate the runtime of a disc of this population for each end time, and see how early and how well the ETA predicts it, to choose abort criteria for the full population.

**Timing CSV columns** (one row per step, or every `stream_timing_every` steps):

| Column | Meaning | Units |
|---|---|---|
| `step` | step number (1 = first step) | – |
| `t_yr` | simulation time at the end of the step | yr |
| `dt_yr` | step actually taken (cut short to land on a snapshot time) | yr |
| `dt_phys_yr` | physics-limited step (min of gas and dust-drift `max_timestep`) | yr |
| `wall_s` | wall time since the time loop started, incl. HDF5 writes (excl. setup before the loop) | s |
| `eta_s` | ETA, step-count estimator: (wall_s / step) × (t_end − t) / dt_phys (the formula behind the printed ETA) | s |
| `eta_rate_s` | ETA, sim-rate estimator: wall_s × (t_end − t) / t | s |

Header lines (`#`) give `t_end_yr`, `stream_timing_every` and `loop_start`. A footer `# loop_end: ..., n_steps: N, wall_s: W` is written when the loop ends, even if it raised. **No footer = the run is still going or was killed** (e.g. by the Slurm time limit, which does not let Python finish). Then the true final runtime is unknown, so the perfect-prediction reference and the ETA error (plots 2 and 3) are skipped for that run.

**Perfect prediction:** with $W$ the final loop wall time (footer `wall_s`), a perfect ETA at elapsed time $w$ is $W - w$.

In [ ]:
# ==== SETTINGS ====
# (Hof, 2026-11-12, sim-time step 3) where the timing CSVs are, and plot options.
import os, glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Folder with the *_timing.csv files (local mount of Nibi's /scratch/imontesd/output/DiscEvolution/...).
# The environment variable HOF_TIMING_DIR overrides it (used to test the notebook on short local runs).
DATADIR_TIMING = os.environ.get("HOF_TIMING_DIR",
                                "/home/imontesd/scratch/output/DiscEvolution/popsynth/add-planets/pilot_v2")
TIMING_GLOB = "*_timing.csv"       # every timing CSV in DATADIR_TIMING is loaded (one curve per run)
TIMING_FILES = None                # or an explicit list of paths, which overrides DATADIR_TIMING + TIMING_GLOB

FIGDIR = os.environ.get("HOF_FIGDIR") or None   # folder to save the figures as PDF; None = do not save
MAX_PLOT_PTS = 200_000             # points drawn per curve (a 5 Myr run has ~5e6 rows); calculations use ALL rows
LOGX_DT = False                    # plot 1: log time axis (shows the early-time dt evolution)
SHOW_RATE_ETA = True               # plot 2: also draw the sim-rate ETA (eta_rate_s) as a dashed line
LOGX_ERR = True                    # plot 3: log step axis (the early steps are where the ETA converges)

## Load the timing CSVs

`load_timing_Hof(path)` returns `(df, meta)`:
- **`df`:** the CSV rows plus derived columns: `t_Myr`, `wall_h`, `eta_h`, `eta_rate_h` and, for a finished run only, `eta_perfect_h` ($W - w$) and `eta_err_h` (ETA − perfect). All in hours except `t_Myr`.
- **`meta`:** `t_end_yr`, `every`, `loop_start`, `complete` (footer present), `n_steps`, `wall_final_s` ($W$, NaN if incomplete), `label` (e.g. "5 Myr").

The table below is one row per run: steps, loop wall time, mean wall time per step, and the first and last physics timestep.

In [ ]:
# ==== LOAD ====
_FOOTER_RE = re.compile(r"#\s*loop_end:.*n_steps:\s*(\d+),\s*wall_s:\s*([0-9.eE+-]+)")


def load_timing_Hof(path):
    # (Hof, 2026-11-12) Read one timing CSV (run_model_Hof.py, sim-time step 1) -> (DataFrame, meta dict). See the
    # markdown above for the columns. Header/footer lines start with "#"; pandas skips them (comment="#") and they are
    # parsed separately here. Without the "# loop_end" footer the run is INCOMPLETE: wall_final_s = NaN and the
    # perfect-prediction columns are not added.
    meta = {"path": path, "t_end_yr": np.nan, "every": 1, "loop_start": "", "complete": False,
            "n_steps": np.nan, "wall_final_s": np.nan}
    with open(path) as fh:
        for line in fh:                                    # header: the first few lines
            if not line.startswith("#"):
                break
            if line.startswith("# t_end_yr:"):
                meta["t_end_yr"] = float(line.split(":", 1)[1])            # yr
            elif line.startswith("# stream_timing_every:"):
                meta["every"] = int(line.split(":", 1)[1])                 # steps per row
            elif line.startswith("# loop_start:"):
                meta["loop_start"] = line.split(":", 1)[1].strip()
    with open(path, "rb") as fh:                           # footer: read only the end of a possibly ~400 MB file
        fh.seek(0, os.SEEK_END)
        fh.seek(max(fh.tell() - 4096, 0))
        m = _FOOTER_RE.search(fh.read().decode(errors="ignore"))
    if m:
        meta["complete"] = True
        meta["n_steps"] = int(m.group(1))
        meta["wall_final_s"] = float(m.group(2))            # s, W = final loop wall time
    df = pd.read_csv(path, comment="#")
    if not meta["complete"] and len(df):
        meta["n_steps"] = int(df["step"].iloc[-1])         # steps so far
    meta["label"] = f"{meta['t_end_yr'] / 1e6:g} Myr"
    df["t_Myr"] = df["t_yr"] / 1e6                          # Myr
    df["wall_h"] = df["wall_s"] / 3600                      # h
    df["eta_h"] = df["eta_s"] / 3600                        # h, step-count ETA
    df["eta_rate_h"] = df["eta_rate_s"] / 3600              # h, sim-rate ETA
    if meta["complete"]:
        df["eta_perfect_h"] = (meta["wall_final_s"] - df["wall_s"]) / 3600   # h, W - w
        df["eta_err_h"] = df["eta_h"] - df["eta_perfect_h"]                  # h, > 0 = ETA too long
    return df, meta


def thin_idx_Hof(n, max_pts=MAX_PLOT_PTS):
    # Row indices to PLOT out of n rows: half evenly spaced in row number, half log-spaced (keeps the early steps,
    # which a linear stride would skip). Plotting only; every calculation uses all rows.
    if n <= max_pts:
        return np.arange(n)
    lin = np.linspace(0, n - 1, max_pts // 2).astype(int)
    log = np.unique(np.geomspace(1, n, max_pts // 2).astype(int)) - 1
    return np.union1d(lin, log)


_files = TIMING_FILES or sorted(glob.glob(os.path.join(DATADIR_TIMING, TIMING_GLOB)))
if not _files:
    raise FileNotFoundError(f"no {TIMING_GLOB} in {DATADIR_TIMING}")
runs = [load_timing_Hof(f) for f in _files]
runs.sort(key=lambda r: r[1]["t_end_yr"])                  # shortest end time first (3 Myr, then 5 Myr)
RUN_COLORS = {m["label"]: f"C{i}" for i, (_, m) in enumerate(runs)}

_rows = []
for df, m in runs:
    _wall = m["wall_final_s"] if m["complete"] else df["wall_s"].iloc[-1]   # s
    _rows.append({"run": m["label"], "complete": m["complete"], "n_steps": m["n_steps"],
                  "t_reached_Myr": df["t_Myr"].iloc[-1],
                  "loop_wall_h": _wall / 3600,
                  "mean_s_per_step": _wall / m["n_steps"],
                  "dt_phys_first_yr": df["dt_phys_yr"].iloc[0], "dt_phys_last_yr": df["dt_phys_yr"].iloc[-1],
                  "csv_rows": len(df), "every": m["every"], "loop_start": m["loop_start"]})
summary_timing = pd.DataFrame(_rows)
print(f"{len(runs)} timing CSV(s) from {DATADIR_TIMING if TIMING_FILES is None else 'TIMING_FILES'}")
for df, m in runs:
    print(f"  {m['label']}: {os.path.basename(m['path'])}" + ("" if m["complete"] else "   (INCOMPLETE: no loop_end footer)"))
with pd.option_context("display.float_format", "{:.4g}".format, "display.width", 200):
    display(summary_timing)

# Check that the runs took the same physics timesteps over the steps they share (expected: identical to 3 Myr).
if len(runs) > 1:
    _a, _b = runs[0][0], runs[-1][0]
    _n = min(len(_a), len(_b)) - 1                          # last row of the shorter run is cut to land on t_end
    _rel = np.max(np.abs(_b["dt_phys_yr"].to_numpy()[:_n] / _a["dt_phys_yr"].to_numpy()[:_n] - 1)) if _n > 0 else np.nan
    print(f"dt_phys, {runs[0][1]['label']} vs {runs[-1][1]['label']} over the first {_n} rows: "
          f"max relative difference {_rel:.2e}")

## Plot 1: timestep vs simulation time

- **Line:** the physics-limited timestep `dt_phys_yr` [yr] against simulation time [Myr], to each run's end time. Up to 3 Myr the runs should overlap exactly (checked in the cell above).
- **Markers:** the steps cut short to land on a snapshot (`dt_yr` < `dt_phys_yr`), at their actual `dt_yr`. These are the only steps where the two runs could differ, and they share all snapshots up to 3 Myr.
- `LOGX_DT = True` (settings cell) gives a log time axis.

In [ ]:
# ==== PLOT 1: dt vs simulation time ====
fig, ax = plt.subplots(figsize=(8, 4.5))
for k, (df, m) in enumerate(runs):
    c, ls = RUN_COLORS[m["label"]], ["-", "--", ":", "-."][k % 4]   # different line style: overlapping runs stay visible
    ii = thin_idx_Hof(len(df))
    ax.plot(df["t_Myr"].to_numpy()[ii], df["dt_phys_yr"].to_numpy()[ii], color=c, ls=ls, lw=1.2,
            label=f"{m['label']}: dt_phys" + ("" if m["complete"] else " (incomplete)"))
    cut = df["dt_yr"] < df["dt_phys_yr"] * (1 - 1e-9)     # steps shortened to hit a snapshot time
    ax.scatter(df.loc[cut, "t_Myr"], df.loc[cut, "dt_yr"], s=14, color=c, marker="v", zorder=3,
               label=f"{m['label']}: step cut to a snapshot ({int(cut.sum())})")
ax.set_xlabel("simulation time (Myr)")
ax.set_ylabel("timestep dt (yr)")
ax.set_yscale("log")                                       # the snapshot-cut steps can be orders of magnitude shorter
if LOGX_DT:
    ax.set_xscale("log")
ax.set_title("Timestep vs simulation time")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
if FIGDIR:
    os.makedirs(FIGDIR, exist_ok=True); fig.savefig(os.path.join(FIGDIR, "simtime_dt_vs_t.pdf"))
plt.show()

## Plot 2: ETA vs elapsed wall time

- **x:** wall time since the loop started, $w$ [h], up to the run's final loop wall time $W$.
- **Solid:** ETA from `eta_s`, the step-count estimator (the printed ETA's formula).
- **Dashed** (`SHOW_RATE_ETA`): ETA from `eta_rate_s`, the sim-rate estimator.
- **Dotted:** perfect prediction $W - w$, a straight line from $(0, W)$ to $(W, 0)$. Only for finished runs.

The setup time before the loop (calibration, initial disc) is not in $w$ or $W$. It is the difference between the run's total duration (end of the log) and $W$.

In [ ]:
# ==== PLOT 2: ETA vs elapsed wall time ====
fig, ax = plt.subplots(figsize=(8, 5))
for df, m in runs:
    c = RUN_COLORS[m["label"]]
    ii = thin_idx_Hof(len(df))
    w = df["wall_h"].to_numpy()[ii]                       # h
    ax.plot(w, df["eta_h"].to_numpy()[ii], color=c, lw=1.3, label=f"{m['label']}: ETA (step-count, eta_s)")
    if SHOW_RATE_ETA:
        ax.plot(w, df["eta_rate_h"].to_numpy()[ii], color=c, lw=1.0, ls="--", alpha=0.7,
                label=f"{m['label']}: ETA (sim-rate, eta_rate_s)")
    if m["complete"]:
        W = m["wall_final_s"] / 3600                       # h
        ax.plot([0, W], [W, 0], color=c, lw=1.2, ls=":", label=f"{m['label']}: perfect, W - w (W = {W:.3g} h)")
    else:
        print(f"{m['label']}: incomplete run, no perfect-prediction line (final runtime unknown)")
ax.set_xlabel("elapsed wall time w (h)")
ax.set_ylabel("ETA, remaining wall time (h)")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.set_title("ETA vs elapsed wall time")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
if FIGDIR:
    os.makedirs(FIGDIR, exist_ok=True); fig.savefig(os.path.join(FIGDIR, "simtime_eta_vs_wall.pdf"))
plt.show()

## Plot 3: ETA error vs step number

- **Line:** ETA − perfect prediction, `eta_err_h` = `eta_h` − $(W - w)$ [h], against step number. Positive means the ETA overestimates the remaining time. Step-count ETA (`eta_s`) only. Finished runs only.
- **Markers:** every x-intercept, i.e. a step where the error changes sign. Each is found by linear interpolation between the two rows either side. The last row is left out: there $t = t_{\rm end}$, so the ETA and the perfect value are both 0 by construction.
- `LOGX_ERR = True` (default) gives a log step axis, which spreads out the early steps where the ETA settles.

The printout lists the number of crossings and the step of the first and last one for each run.

In [ ]:
# ==== PLOT 3: ETA - perfect prediction vs step number ====
def zero_crossings_Hof(x, y):
    # x positions where y changes sign, linearly interpolated between neighbouring rows; exact zeros count once.
    x, y = np.asarray(x, float), np.asarray(y, float)
    s = np.sign(y)
    xs = list(x[s == 0])                                   # rows exactly on zero
    i = np.flatnonzero(s[:-1] * s[1:] < 0)                 # sign change between row i and i + 1
    xs += list(x[i] + (x[i + 1] - x[i]) * y[i] / (y[i] - y[i + 1]))
    return np.sort(np.array(xs))


fig, ax = plt.subplots(figsize=(8, 4.5))
for df, m in runs:
    if not m["complete"]:
        print(f"{m['label']}: incomplete run, skipped (final runtime unknown)")
        continue
    c = RUN_COLORS[m["label"]]
    step = df["step"].to_numpy()[:-1]                      # last row dropped: error = 0 - 0 by construction
    err = df["eta_err_h"].to_numpy()[:-1]                  # h
    xz = zero_crossings_Hof(step, err)
    ii = thin_idx_Hof(len(step))
    ax.plot(step[ii], err[ii], color=c, lw=1.1, label=f"{m['label']}: ETA − perfect")
    ax.scatter(xz, np.zeros_like(xz), s=30, facecolors="none", edgecolors=c, linewidths=1.2, zorder=3,
               label=f"{m['label']}: x-intercepts ({len(xz)})")
    if len(xz):
        print(f"{m['label']}: {len(xz)} x-intercept(s); first at step {xz[0]:.6g}, last at step {xz[-1]:.6g} "
              f"(of {m['n_steps']} steps)")
    else:
        print(f"{m['label']}: no x-intercept (ETA error keeps one sign over the whole run)")
ax.axhline(0, color="0.4", lw=0.8)
ax.set_xlabel("step number")
ax.set_ylabel("ETA − perfect prediction (h)")
if LOGX_ERR:
    ax.set_xscale("log")
ax.set_title("ETA error vs step number")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
if FIGDIR:
    os.makedirs(FIGDIR, exist_ok=True); fig.savefig(os.path.join(FIGDIR, "simtime_eta_err_vs_step.pdf"))
plt.show()